In [4]:
import sys
import os
sys.path.append("../")
sys.path.append("../../")

In [5]:
from scale_rl.common.wandb_utils import *

In [13]:
from scale_rl.envs.dmc import DMC_EASY_MEDIUM

def replace_hypen_to_underbar(env_name_list):
    for idx in range(len(env_name_list)):
        env_name_list[idx] = env_name_list[idx].replace('_', '-')
    return env_name_list

DMC_EASY_MEDIUM = replace_hypen_to_underbar(DMC_EASY_MEDIUM)

sac_simba_eval_df = pd.read_csv('../results/baseline/simba.csv', index_col=0)
sac_simba_eval_df['env_name'] = sac_simba_eval_df['env_name'].str.replace('_', '-')
DMC_EASY_MEDIUM
sac_simba_eval_df = sac_simba_eval_df[~(sac_simba_eval_df['env_name'].isin(DMC_EASY_MEDIUM))]

In [14]:
entity = 'draftrec'
project_name = 'Simba_2412'
run_exp_names_to_analysis_exp_names = {
    'sac_simba': 'sac_simba',
}
run_exp_names = list(run_exp_names_to_analysis_exp_names.keys())
metrics = ['avg_return', 'avg_success']

In [15]:
runs = collect_runs(entity=entity, project_name=project_name) 
filtered_runs = filter_runs(runs, exp_names = run_exp_names)
wandb_df = convert_runs_to_dataframe(
    runs = filtered_runs, 
    run_exp_name_to_analysis_exp_name=run_exp_names_to_analysis_exp_names
)
wandb_df = wandb_df[wandb_df.apply(lambda row: 'finished' in str(row['run']), axis=1)]
eval_df = convert_wandb_df_to_eval_df(wandb_df, metrics)
eval_df['env_name'] = eval_df['env_name'].str.replace('_', '-')
eval_df


  0%|          | 0/2830 [00:00<?, ?it/s]

  0%|          | 0/260 [00:00<?, ?it/s]

  0%|          | 0/260 [00:00<?, ?it/s]

,exp_name,env_name,seed,metric,env_step,value
0,sac_simba,Humanoid-v4,9000,avg_return,0.0,158.084484
1,sac_simba,Humanoid-v4,9000,avg_return,50000.0,919.968914
2,sac_simba,Humanoid-v4,9000,avg_return,100000.0,6104.829226
3,sac_simba,Humanoid-v4,9000,avg_return,150000.0,2061.806108
4,sac_simba,Humanoid-v4,9000,avg_return,200000.0,5229.141110
...,...,...,...,...,...,...
6715,sac_simba,acrobot-swingup,0,avg_success,600000.0,0.000000
6716,sac_simba,acrobot-swingup,0,avg_success,700000.0,0.000000
6717,sac_simba,acrobot-swingup,0,avg_success,800000.0,0.000000
6718,sac_simba,acrobot-swingup,0,avg_success,900000.0,0.000000


In [16]:
def update_seeds_without_value_check(eval_df):
    """
    Updates the seed values in eval_df when exp_name, env_name, and env_step are the same.

    Args:
    - eval_df (pandas DataFrame): DataFrame containing the evaluation data.

    Returns:
    - eval_df (pandas DataFrame): Updated DataFrame with non-duplicate seed values for 
                                  matching exp_name, env_name, and env_step.
    """
    # Create a copy of the DataFrame to avoid modifying the original
    updated_df = eval_df.copy()

    # Track the seeds that have been used for each (exp_name, env_name, env_step)
    seed_tracker = {}

    # Iterate over the DataFrame row by row
    for idx, row in updated_df.iterrows():
        exp_name = row["exp_name"]
        env_name = row["env_name"]
        env_step = row["env_step"]
        metric = row['metric']
        seed = row["seed"]

        key = (exp_name, env_name, env_step, metric)

        if key not in seed_tracker:
            seed_tracker[key] = set()

        # If the current seed has already been used for this key (exp_name, env_name, env_step)
        if seed in seed_tracker[key]:
            # Increment the seed until we find a unique one
            new_seed = seed + 500
            while new_seed in seed_tracker[key]:
                new_seed += 500
            # Update the seed in the DataFrame
            updated_df.at[idx, "seed"] = new_seed
            # Add the new seed to the tracker
            seed_tracker[key].add(new_seed)
        else:
            # Add the current seed to the tracker
            seed_tracker[key].add(seed)

    return updated_df

In [17]:
eval_df = pd.concat([eval_df, sac_simba_eval_df], ignore_index=True)
# new_df = update_seeds_without_value_check(eval_df)

In [18]:
eval_df.to_csv("/home/nas4_user/youngdolee/SimbaV2/results/1219_sac_simba_1m.csv")